# Southwest Q3 2026 forecast

**Data cutoff:** October 16, 2026  
**Target:** Q3 2026 revenue passengers and average passenger fare  
**Evaluation:** walk-forward forecasts from 2022Q1 through 2026Q2

This notebook presents the auditable results. Reusable data preparation, forecasting, calibration, and validation live in `src/southwest_forecast.py`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

root = Path.cwd().resolve()
if not (root / 'Case Study Data Files').is_dir():
    root = root.parent
sys.path.insert(0, str(root))

from src.southwest_forecast import run_analysis
result = run_analysis(root)

## Cleaning decisions and disruption flags

TSA counts are retained as reported. Known disruption dates are flagged and affected passenger residuals are excluded from interval calibration. A BLS quarter is accepted only when all three monthly values are present.

In [ ]:
display(result['cleaning_log'])
flag_columns = ['tsa_total', 'tsa_days', 'tsa_disruption_days', 'tsa_clean_for_calibration', 'airline_cpi', 'cpi_months', 'cpi_complete']
display(result['data'].loc['2025Q3':'2026Q3', flag_columns].round(3))

## Explainable point model

For passengers, Q3 2025 is scaled by an equal-weight blend of Southwest's latest passenger growth and Q3 TSA growth. For fare, Q3 2025 is scaled by an equal-weight blend of Southwest's latest fare growth and Q3 BLS airline-fare CPI growth. The 50/50 weights are fixed, not optimized on the backtest.

## Walk-forward backtest and baselines

Each row uses only lagged Southwest observations and the public signal available for that quarter. The table includes the two required baselines and labels the business-model regime.

In [ ]:
columns = [
    'regime', 'tsa_disruption_days', 'cpi_complete',
    'actual_passengers', 'same_year_ago_passengers', 'company_growth_passengers', 'blend_passengers',
    'actual_fare', 'same_year_ago_fare', 'company_growth_fare', 'blend_fare',
]
display(result['backtest'][columns].round(3))

In [ ]:
summary = result['backtest_summary'].pivot_table(
    index=['regime', 'target'], columns='model', values=['n', 'mape_pct'], aggfunc='first'
)
display(summary.round(2))

## Rolling conformal intervals

Intervals use the finite-sample 80% absolute-residual quantile from at most the latest 12 valid blend residuals. Passenger calibration excludes quarters containing flagged TSA disruption days; fare calibration excludes incomplete CPI quarters. Coverage is evaluated sequentially using only residuals available before each test quarter.

In [ ]:
display(result['forecast'].round(3))

## T-100 July cross-check

T-100 is not inserted into the point formula because it covers one domestic month and counts flight segments rather than one-way revenue journeys. It is reported as a directional challenge to the TSA-based result.

In [ ]:
display(result['t100_cross_check'].round(3))

## Final submission and guidance reconciliation

- Revenue passengers: **33.516 million**, 80% interval **31.444–35.587 million**
- Average passenger fare: **$223.32**, 80% interval **$205.32–$241.32**
- Point passenger revenue: **$7.485 billion**
- Guidance-implied passenger revenue: **$7.437–$7.640 billion**

The point revenue forecast falls inside the guidance-implied range. The mechanical product of marginal interval endpoints is shown below for transparency, but it is not a calibrated joint 80% revenue interval.

In [ ]:
display(result['reconciliation'].round(3))